# 04 — Evaluation: Holdout Test, Explainability, Simulated Transfer

Held-out 30-building test split (building-disjoint, D-002), SHAP explanation,
and a **SIMULATED** geothermal transfer smoke test. Reproduce:
`python scripts/run_all.py` (stages 9–13).

In [ ]:
import json
ht = json.load(open('../results/metrics/holdout_test_final.json'))
print(f"Test set          : {ht['n']:,} rows, {ht['n_buildings']} buildings, base rate {ht['base_rate']:.2%}")
print(f"ROC-AUC           : {ht['roc_auc']:.4f} (95% CI {ht['auc_ci95_low']:.4f}-{ht['auc_ci95_high']:.4f}, {ht['n_boot_valid']} bootstrap)")
print(f"PR-AUC            : {ht['pr_auc']:.4f}")
print(f"Brier             : {ht['brier']:.5f} (ECE {ht['ece_uniform_15bin']:.5f})")
print(f"Episode recall    : {ht['event_recall']:.4f} ({ht['events_detected']}/{ht['n_events']})")
print(f"Median latency    : {ht['median_detection_latency_h']:.0f} h")
print(f"False alarms      : {ht['false_alarms_per_building_day']:.3f} per building-day")

In [ ]:
# Global explanation — top SHAP features
sh = json.load(open('../results/metrics/shap_summary_final.json'))
top = sh['global_top15'][:10]
names = [f['feature'] for f in top][::-1]
vals  = [f['mean_abs_shap'] for f in top][::-1]
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.barh(names, vals, color='#c0504d')
plt.xlabel('mean |SHAP|')
plt.title(f"Top-10 features ({sh['method']}, n={sh['n_explained']:,})")
plt.tight_layout(); plt.show()

In [ ]:
# SIMULATED geothermal transfer test — pipeline smoke test, NOT plant validation
sim = json.load(open('../results/metrics/simulation_detection_SIMULATED.json'))
print(sim['warning'])
print()
print(f"{'equipment':<11}{'ROC-AUC':>9}{'PR-AUC':>9}{'fault_recall':>14}")
for r in sim['results']:
    print(f"{r['equipment']:<11}{r['roc_auc']:>9.4f}{r['pr_auc']:>9.4f}{r['fault_recall']:>14.4f}")
print()
print(sim['conclusion'])

**Interpretation:** held-out performance is strong and calibrated; the frozen
threshold does **not** transfer to a different asset class without
recalibration. Geothermal numbers are from synthetic signals only (D-005).